1. Загрузите картинку parrots.jpg. Преобразуйте изображение, приведя все значения в интервал от 0 до 1. Для этого можно воспользоваться функцией img_as_float из модуля skimage. Обратите внимание на этот шаг, так как при работе с исходным изображением
вы получите некорректный результат.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from skimage.color import rgb2gray
from sklearn.cluster import KMeans
from skimage.util import img_as_float

image = io.imread('parrots.jpg')
image = img_as_float(image)

2. Создайте матрицу объекты-признаки: характеризуйте каждый пиксель тремя координатами - значениями интенсивности в пространстве RGB.

In [5]:
h, w, c = image.shape
pixels = image.reshape(-1, c)

3. Запустите алгоритм K-Means с параметрами init=’k-means++’ и
random_state=241. После выделения кластеров все пиксели, отнесенные в один кластер, попробуйте заполнить двумя способами:
медианным и средним цветом по кластеру.


4. Измерьте качество получившейся сегментации с помощью метрики
PSNR. Эту метрику нужно реализовать самостоятельно (см. определение).

In [6]:
def psnr(original, compressed):
    mse = np.mean((original - compressed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    return 10 * np.log10(max_pixel**2 / mse)
results = {}
max_k = 15
for k in range(1, max_k + 1):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=241, n_init=10)
    labels = kmeans.fit_predict(pixels)
    img_median = np.zeros_like(pixels)
    for cluster in range(k):
        mask = (labels == cluster)
        if np.any(mask):
            median_color = np.median(pixels[mask], axis=0)
            img_median[mask] = median_color
    img_median = img_median.reshape(h, w, c)

    img_mean = np.zeros_like(pixels)
    for cluster in range(k):
        mask = (labels == cluster)
        if np.any(mask):
            mean_color = np.mean(pixels[mask], axis=0)
            img_mean[mask] = mean_color
    img_mean = img_mean.reshape(h, w, c)

    psnr_med = psnr(image, img_median)
    psnr_mean = psnr(image, img_mean)
    results[k] = (psnr_med, psnr_mean)
    print(f"  k={k}: PSNR_median = {psnr_med:.2f}, PSNR_mean = {psnr_mean:.2f}")

  k=1: PSNR_median = 9.46, PSNR_mean = 9.84
  k=2: PSNR_median = 11.68, PSNR_mean = 12.11
  k=3: PSNR_median = 12.80, PSNR_mean = 13.18
  k=4: PSNR_median = 14.04, PSNR_mean = 14.39
  k=5: PSNR_median = 15.21, PSNR_mean = 15.56
  k=6: PSNR_median = 16.08, PSNR_mean = 16.57
  k=7: PSNR_median = 17.37, PSNR_mean = 17.67
  k=8: PSNR_median = 18.18, PSNR_mean = 18.47
  k=9: PSNR_median = 18.85, PSNR_mean = 19.14
  k=10: PSNR_median = 19.39, PSNR_mean = 19.67
  k=11: PSNR_median = 19.89, PSNR_mean = 20.16
  k=12: PSNR_median = 20.34, PSNR_mean = 20.63
  k=13: PSNR_median = 20.83, PSNR_mean = 21.06
  k=14: PSNR_median = 21.16, PSNR_mean = 21.37
  k=15: PSNR_median = 21.45, PSNR_mean = 21.65


5. Найдите минимальное количество кластеров, при котором значение PSNR выше 20 (можно рассмотреть не более 20 кластеров, но
не забудьте рассмотреть оба способа заполнения пикселей одного
кластера). Это число и будет ответом в данной задаче.

In [9]:
answer = None
for k in range(1, max_k + 1):
    psnr_med, psnr_mean = results[k]
    if psnr_med > 20 and psnr_mean > 20:
        answer = k
        break
print(f"\n{answer}")
with open('result14.txt','w',encoding='utf-8') as f:
    f.write(f"{psnr_med:.2f} {psnr_mean:.2f}\n{answer}")


12
